# 08 - SymptoGraph-LLM v2

A revision of the proposed model in response to review. Components were
**removed** as well as added, and every addition either reduces effective
capacity or corrects supervision - because the model's problem is not capacity.

## Why not "add a bigger module"

Training risk QWK reaches **1.000** and training severity QWK **1.000** by epoch
6, while validation sits near 0.40 and 0.47 and validation loss starts rising
from epoch 2. A model that already fits its training set perfectly is not short
of representational power; it is short of **sample efficiency**. Anything that
adds parameters makes this worse.

## Removed

| component | why |
|---|---|
| **N5 contrastive** | 0.544 -> 0.537 across two full runs; at or below seed noise, and it costs a projection head plus a loss term |
| **N6 uncertainty** | measured effect -0.003; the hypothesis is already recorded as wrong in `Model_Design.md` 3.4 |
| **N2 graph** (demoted) | L1 -> L2 is 0.549 -> 0.571 with sds 0.027 and 0.015, so it does not clear seed noise, and the prior is hand-authored. Kept as control `C_graph`, not as a headline novelty |

## Added

**N-A - instrument-grounded monotone ordinal head.** PHQ-9 severity is defined
as a threshold function of the sum of its nine item scores, and 9 of the 22
symptoms *are* those items (7 more are GAD-7). The old free `Linear(head_in, 4)`
severity head discarded that. Instead:

```
z      = sum_k softplus(v_k) * sigma(c_k)        # one latent distress trait
P(y>=j) = sigma(z - tau_j),  tau_1 < ... < tau_J  # cumulative link per task
```

Suicide risk now fits **4 cutpoints** on its 340 rows instead of a whole head,
and borrows strength from the 2,456 severity rows through `z`. Non-negative item
weights make severity provably monotone in symptom presence - a hard constraint,
not a learned preference. The learned weights are readable as item loadings.

**N-B - per-symptom labeller noise channel.** A single confidence temperature
rescales everything equally, so it cannot fix the labeller calling
`low_self_esteem` at 0.80 where gold prevalence is 0.24. Section 3 fits
`P(noisy | clean)` per symptom from the gold-labelled symptoms, and training
pushes the clean probability through that channel before scoring it.

**N-C - author-consistency self-supervision:** *not implemented here.* It needs
authors with multiple posts; the pool averages 1.34 posts/author and I have not
measured the distribution. Section 2b measures it. If the support is not there,
N-C is dropped rather than shipped on an assumption.

## Overfitting controls (all rungs, identically)

| control | reason |
|---|---|
| `epochs = 4` | validation loss bottoms out at epoch 1-2 in every seed |
| `selection_metric = 'neg_val_loss'` | `selection_score` is 2/3 symptom metrics, which keep climbing while the ordinal heads memorise; it picked epoch 10 at ~2x minimum validation loss on two seeds |
| `early_stopping_patience = 2` | matched to where the minimum actually is |
| `label_smoothing = 0.05` | on the ordinal heads only |
| the monotone head itself | the largest structural regulariser, by construction |

## What this does NOT fix

Of the six review actions, this notebook addresses overfitting and the labeller
bias. It does **not** fix the baseline comparison (notebook 05 must be re-run
with real MentalRoBERTa - that is the most serious comment and no claim here
survives without it), the missing in-house gold set, or the clinician review of
the prior. **Run the baselines first.**

## 1. Setup

In [ ]:
import os, sys, json, math, warnings
import numpy as np, pandas as pd
from google.colab import drive
warnings.filterwarnings('ignore')
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/hari/SymptoGraph-LLM'
PROCESSED  = f'{DRIVE_BASE}/processed'
GRAPH      = f'{DRIVE_BASE}/graph'
WEAK_DIR   = f'{DRIVE_BASE}/weak_labels'
WEAK       = f'{WEAK_DIR}/weak_labels.parquet'
VALIDATION = f'{WEAK_DIR}/validation_labels.jsonl'
NOISE      = f'{WEAK_DIR}/noise_channel.json'
BASELINES_DIR = f'{DRIVE_BASE}/baselines'
RESULTS    = f'{DRIVE_BASE}/proposed_v2'

RUNGS    = None    # None = all five
CONTROLS = []      # e.g. ['C_graph'] once the ladder is done

os.makedirs(RESULTS, exist_ok=True)
for p in (PROCESSED, GRAPH):
    assert os.path.isdir(p), f'missing {p}'
assert os.path.exists(WEAK), f'missing {WEAK} - run notebook 04'
print('writing to', RESULTS)


In [ ]:
REPO, CHECKOUT = 'https://github.com/sid-2k6/SymptoGraph-LLM.git', '/content/SymptoGraph-LLM'
if os.path.isdir(f'{CHECKOUT}/.git'):
    !git -C $CHECKOUT fetch -q origin && git -C $CHECKOUT reset -q --hard origin/main
else:
    !git clone -q --branch main $REPO $CHECKOUT
sys.path.insert(0, f'{CHECKOUT}/src')
from symptograph import proposed as P
assert hasattr(P, 'LADDER_V2'), 'stale checkout - !rm -rf /content/SymptoGraph-LLM'
!git -C $CHECKOUT log --oneline -1

import torch
assert torch.cuda.is_available(), 'need a GPU runtime'
print(torch.cuda.get_device_name(0),
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')


## 2b. Is novelty N-C even possible?

Author-consistency needs authors with more than one post. The pool averages 1.34
posts per author, so this measures the distribution before any code is written
for it. **No GPU.**

In [ ]:
import pyarrow.parquet as pq
counts = {}
f = pq.ParquetFile(f'{PROCESSED}/weak_pool.parquet')
for batch in f.iter_batches(batch_size=200_000, columns=['author_hash']):
    for a in batch.column('author_hash').to_pylist():
        if a:
            counts[a] = counts.get(a, 0) + 1
c = pd.Series(counts)
print(f'{len(c):,} known authors, {int(c.sum()):,} posts')
print(f'posts/author: mean {c.mean():.2f}  median {c.median():.0f}  max {c.max():,}')
dist = c.value_counts().sort_index()
print('\nauthors by post count:')
print(dist.head(8).to_string())
multi = c[c >= 2]
print(f'\nauthors with >=2 posts : {len(multi):,} ({100*len(multi)/len(c):.1f}%)')
print(f'posts held by them     : {int(multi.sum()):,}')
if len(multi) < 20_000:
    print('\nN-C NOT VIABLE - too few multi-post authors. Dropping it is the')
    print('correct call; do not implement it on an assumption.')
else:
    print('\nN-C is viable on this subset. Still needs its own compute budget;')
    print('it is not part of the ladder below.')


## 3. Fit the labeller noise channel (N-B)

Uses `validation_labels.jsonl` - the labeller run over the symptoms that *do*
have gold labels. **No GPU.** The output table is itself a review deliverable:
it quantifies the per-symptom bias your superior flagged.

Symptoms with no gold labels cannot be measured; they are pooled by instrument
and marked as such. That is an assumption the paper must state.

In [ ]:
if not os.path.exists(VALIDATION):
    raise FileNotFoundError(
        f'{VALIDATION} not found. It is written by notebook 04 section 5. '
        'Without it N-B cannot be fitted - run M0-M3 and leave M4 out.')
!python $CHECKOUT/scripts/fit_noise_channel.py \
    --validation "$VALIDATION" --labelled "$PROCESSED/labelled.parquet" \
    --out "$NOISE" --ontology $CHECKOUT/configs/symptom_ontology.json


## 4. Configuration

In [ ]:
from symptograph.proposed import ProposedConfig, LADDER_V2, LADDER_V2_ORDER, \
                                   FULL_MODEL_V2, CONTROLS_V2, check_encoder_access
from symptograph.preprocess import Ontology
from symptograph.metrics import compare_runs, headline_table, set_publication_style
from symptograph.jsonio import read_json, write_json

onto = Ontology.load(f'{CHECKOUT}/configs/symptom_ontology.json')
set_publication_style(font_size=20)
ENCODER = 'mental/mental-roberta-base'
st = check_encoder_access(ENCODER)
if not st['ok']:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    st = check_encoder_access(ENCODER)
assert st['ok'], st['reason']

cfg = ProposedConfig(
    epochs=4, seeds=(0, 1, 2),
    batch_size=32, eval_batch_size=64, max_length=192,
    encoder_name=ENCODER, strict_encoder=True,
    # --- overfitting controls
    selection_metric='neg_val_loss',
    early_stopping_patience=2,
    label_smoothing=0.05,
    scheduler='cosine',
    freeze_encoder_layers=0,     # try 4 if epoch 2 still overfits
    # --- retained from v1
    pos_weight_cap=10.0, ordinal_class_weights=True, tune_thresholds=True,
    concept_gated_pooling=True,
    # --- weak supervision: stream, all ~50k labels
    weak_label_path=WEAK, weak_as_stream=True, weak_max_rows=None,
    weak_batch_size=16, weak_every=1,
    # --- N-B
    noise_channel_path=NOISE,
    amp_dtype='auto', checkpoint_every=1)

print(f'epochs {cfg.epochs} | seeds {cfg.seeds} | batch {cfg.batch_size}')
print(f'selection {cfg.selection_metric} | patience {cfg.early_stopping_patience}'
      f' | label smoothing {cfg.label_smoothing}')
print(f'frozen encoder blocks {cfg.freeze_encoder_layers}')
print()
for n, f in LADDER_V2:
    on = [k.replace('use_','') for k, v in f.items() if v]
    print(f'  {n:16s} {", ".join(on) or "(none)"}')
print('\ncontrols available:', [n for n, _ in CONTROLS_V2])


## 5. Pre-registered predictions

Written before training. The point is that each one can fail and still be
reportable.

In [ ]:
H = {
 'removed': {'contrastive': '0.544 -> 0.537 over two runs, within seed noise',
             'uncertainty': 'measured -0.003; hypothesis already recorded wrong',
             'graph': 'L1->L2 within seed noise; demoted to control C_graph'},
 'predictions': [
   'N-A raises risk QWK most, because risk goes from a full head to 4 cutpoints '
   'on 340 rows while z is trained by 2,456 severity rows',
   'N-A may slightly LOWER severity QWK if the true relationship is not monotone '
   'in any weighted sum of symptoms; C_monotone_unconstrained separates '
   '"structure helps" from "fewer parameters help"',
   'epochs=4 plus neg_val_loss selection removes the >1.3x validation-loss '
   'inflation seen at the selected epoch in 24 of 27 earlier runs',
   'N-B helps the gold-labelled symptoms, where the channel is measured; its '
   'effect on the 11 weak-only symptoms is NOT measurable',
 ],
 'not_predicted': [
   'any metric reaching 0.85 beyond stress AUROC and symptom AUROC, which '
   'already exceed it',
   'a guaranteed margin for the full model over every rung',
   'anything about risk QWK differences smaller than ~0.4, which is the width '
   'of its 95% bootstrap CI on 74 test rows',
 ],
 'unfixed_by_this_notebook': [
   'baseline comparison - notebook 05 must be re-run with real MentalRoBERTa',
   'N3 validation - needs the in-house anxiety gold set',
   'clinician review of the DSM-5 prior',
 ]}
write_json(f'{RESULTS}/hypothesis_v2.json', H)
print(json.dumps(H, indent=2))


## 6. Train

5 rungs x 3 seeds x 4 epochs. Roughly **1.5 h** on an L4: M0-M2 have no weak
stream (~82 s/epoch), M3-M4 do (~105 s/epoch). Resumable; set `RUNGS` in cell 1
to split sessions.

In [ ]:
results = P.run_ladder(cfg, PROCESSED, GRAPH, RESULTS, rungs=RUNGS,
                       ladder=LADDER_V2, onto=onto, resume=True, verbose=True)
print('\ntrained:', list(results))


## 7. Did the overfitting actually stop?

The first thing to check, before any metric. If `inflation` is still above ~1.3
the selection fix did not work and the numbers below are not trustworthy.

In [ ]:
rows = []
for rung in LADDER_V2_ORDER:
    for s in cfg.seeds:
        p = f'{RESULTS}/{rung}/seed_{s}/history.csv'
        if not os.path.isfile(p):
            continue
        h = pd.read_csv(p)
        sel = h.loc[h.val_loss.idxmin()]
        rows.append({'rung': rung, 'seed': s, 'epochs_run': len(h),
                     'ep_min_val_loss': int(sel.epoch),
                     'val_loss_min': round(h.val_loss.min(), 3),
                     'val_loss_last': round(h.val_loss.iloc[-1], 3),
                     'inflation_at_last': round(h.val_loss.iloc[-1]/h.val_loss.min(), 2),
                     'train_sevQWK_last': round(h.train_severity_depression_qwk.iloc[-1], 3),
                     'train_riskQWK_last': round(h.train_severity_risk_qwk.iloc[-1], 3)})
ov = pd.DataFrame(rows)
display(ov.set_index(['rung', 'seed']))
mem = ov[ov.train_riskQWK_last > 0.95]
print(f'runs still memorising the risk split (train QWK > 0.95): {len(mem)}/{len(ov)}')
if len(mem):
    print('Raise freeze_encoder_layers to 4, or drop to epochs=3, and re-run.')
else:
    print('No run memorised the risk split. Compare against 27/27 before.')


## 8. The monotone head, read as item loadings

The interpretability payoff of N-A: PHQ-9 scores every item equally, so a learned
weighting that departs from uniform is a concrete, checkable clinical claim.

In [ ]:
import matplotlib.pyplot as plt
import torch

ck = torch.load(f'{RESULTS}/{FULL_MODEL_V2}/seed_0/checkpoint_best.pt',
                map_location='cpu', weights_only=False)
sd = ck['model']
if 'item_w' in sd:
    W = torch.nn.functional.softplus(sd['item_w'])
    if W.dim() == 1:
        W = W.unsqueeze(0)
    traits = ['shared'] if W.shape[0] == 1 else ['severity', 'risk']
    for ti, tname in enumerate(traits):
        w = W[ti].numpy()
        w = w / w.sum() * len(w)        # 1.0 == PHQ-9's equal weighting
        t = pd.DataFrame({'symptom': onto.symptoms,
                          'instrument': [onto.instrument(s) for s in onto.symptoms],
                          'relative_weight': w.round(3)}).sort_values(
                              'relative_weight', ascending=False)
        print(f'=== trait: {tname} ===')
        display(t.set_index('symptom').head(10))
        fig, ax = plt.subplots(figsize=(10, 7))
        col = {'PHQ-9': '#2b6cb0', 'GAD-7': '#c05621', 'extra': '#4a5568'}
        ax.barh(t.symptom[::-1], t.relative_weight[::-1],
                color=[col[i] for i in t.instrument[::-1]])
        ax.axvline(1.0, ls='--', c='k', lw=1.5)
        ax.set_xlabel('weight relative to equal (PHQ-9) weighting')
        ax.set_title(f'N-A learned item loadings ({tname})')
        fig.tight_layout()
        fig.savefig(f'{RESULTS}/item_loadings_{tname}.png', dpi=300,
                    bbox_inches='tight')
        plt.show()
    for key in ('severity', 'risk'):
        inc = torch.nn.functional.softplus(sd[f'cut_inc.{key}']) + 1e-3
        tau = torch.cat([sd[f'cut0.{key}'], sd[f'cut0.{key}'] + inc.cumsum(0)])
        print(f'{key:9s} cutpoints: {[round(float(x), 3) for x in tau]}')
else:
    print('no monotone head in this checkpoint')

## 9. Ladder, with bootstrap confidence intervals

In [ ]:
runs = {r: f'{RESULTS}/{r}' for r in LADDER_V2_ORDER
        if os.path.isfile(f'{RESULTS}/{r}/aggregate/aggregate_metrics.json')}
combined, tables = compare_runs(runs, f'{RESULTS}/comparison_ladder',
                                title='SymptoGraph-LLM v2 ablation ladder',
                                font_size=20)
display(combined.pivot(index='metric', columns='model', values='formatted'))


In [ ]:
# Per-task 95% CIs over resampled TEST EXAMPLES, model held fixed.
# This is the variance that matters for risk: 74 rows.
from symptograph.significance import load_predictions, bootstrap_ci, metric_fns

fns = metric_fns()
rows = []
for rung in runs:
    thr = read_json(f'{RESULTS}/{rung}/seed_0/performance_metrics.json').get('thresholds')
    thr_arr = None if not thr else np.array([thr[s] for s in onto.symptoms])
    f_thr = metric_fns(thr_arr)
    for name, fn in f_thr.items():
        pts = []
        for s in cfg.seeds:
            d = load_predictions(f'{RESULTS}/{rung}', s, 'test')
            if d is None:
                continue
            pts.append(bootstrap_ci(d, fn, n_boot=1000, seed=s))
        if pts:
            rows.append({'rung': rung, 'metric': name,
                         'mean': round(float(np.mean([p['point'] for p in pts])), 4),
                         'seed_std': round(float(np.std([p['point'] for p in pts], ddof=1)), 4),
                         'ci_lo': round(float(np.mean([p['lo'] for p in pts])), 4),
                         'ci_hi': round(float(np.mean([p['hi'] for p in pts])), 4),
                         'ci_width': round(float(np.mean([p['hi']-p['lo'] for p in pts])), 4),
                         'n_test': pts[0]['n']})
ci = pd.DataFrame(rows)
display(ci.pivot(index='metric', columns='rung',
                 values='mean').join(
        ci.groupby('metric').ci_width.mean().round(3).rename('mean_ci_width')))
ci.to_csv(f'{RESULTS}/comparison_ladder/bootstrap_cis.csv', index=False)
print('A rung-to-rung gap narrower than ci_width is not measurable on this test set.')


## 10. Paired significance tests

Resamples the **same** test examples for both models, so the comparison is
paired. A difference counts only if it clears p < 0.05 on every seed *and* has
the same sign on every seed - a per-seed p-value alone ignores training
variance.

In [ ]:
from symptograph.significance import compare_models

order = [r for r in LADDER_V2_ORDER if r in runs]
out = []
for a, b in zip(order, order[1:]):
    for row in compare_models(f'{RESULTS}/{a}', f'{RESULTS}/{b}',
                              seeds=cfg.seeds, split='test', n_boot=1000,
                              label_a=a, label_b=b):
        row['step'] = f'{a} -> {b}'
        out.append(row)
sig = pd.DataFrame(out)
cols = ['step', 'metric'] + [c for c in sig.columns if c not in ('step', 'metric')]
display(sig[cols].set_index(['step', 'metric']))
sig.to_csv(f'{RESULTS}/comparison_ladder/paired_bootstrap.csv', index=False)
n = int(sig.significant.sum())
print(f'{n} of {len(sig)} rung-to-rung differences are significant and '
      f'sign-consistent across seeds.')
print('Report the non-significant ones as non-significant. That is the finding.')


## 11. Controls

Run these after the ladder. Each isolates one claim, and each costs 3 seeds
x 4 epochs (~25 min for all three).

- `C_graph` - does the graph beat no graph on top of the full model?
- `C_monotone_unconstrained` - is N-A winning from structure or just fewer parameters?
- `C_weak_downweight` - does denoising beat simply halving the weak-label weight?

In [ ]:
if CONTROLS:
    for name, flags in CONTROLS_V2:
        if name not in CONTROLS:
            continue
        P.run_rung(name, flags, cfg, PROCESSED, GRAPH, RESULTS,
                   onto=onto, resume=True, verbose=True)
    for name in CONTROLS:
        f = f'{RESULTS}/{name}/aggregate/aggregate_metrics.json'
        if os.path.isfile(f):
            agg = read_json(f)['aggregate']
            full = read_json(f'{RESULTS}/{FULL_MODEL_V2}/aggregate/'
                             'aggregate_metrics.json')['aggregate']
            print(f'\n{name} vs {FULL_MODEL_V2}:')
            for m in ('composite', 'severity_depression_qwk', 'severity_risk_qwk',
                      'symptom_macro_auroc[all]'):
                if m in agg and m in full:
                    print(f'  {m:34s} {agg[m]["mean"]:.4f} vs {full[m]["mean"]:.4f}'
                          f'  ({agg[m]["mean"]-full[m]["mean"]:+.4f})')
else:
    print('Set CONTROLS in cell 1, e.g. CONTROLS = [\'C_graph\'].')


## 12. Reading this honestly

**Check section 7 first.** If runs are still memorising the risk split, nothing
else here is trustworthy.

**Sections 9 and 10 are what answer the review.** Report per-task means with CIs
and the paired test result, not a composite. If a rung-to-rung gap is inside the
CI width, say so - the 74-row risk split has a 95% CI roughly **0.4 wide**, which
means risk QWK cannot settle any architectural question on this data. That is a
property of the dataset, not a failure of the model, and stating it plainly is
stronger than reporting a number that cannot be defended.

**If N-A does not help**, `C_monotone_unconstrained` tells you whether the
monotone constraint or merely the parameter reduction was responsible, and that
is a publishable result either way.

**Still required before submission**, and none of it is architectural:

1. Re-run notebook 05 with real MentalRoBERTa and this protocol. `M0_flat` is
   the fair internal baseline and should be the headline comparison.
2. The 800-post in-house anxiety gold set - the only way N3 and the weak-only
   half of N-B become measurable.
3. Clinician review of the symptom-disorder prior, or a sensitivity test over
   perturbed priors.
4. A matched-data baseline: the baselines never saw the ~50k weak labels, so
   part of any margin is extra supervision rather than architecture.